# Capítulo 8: Regressão Linear

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 3 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> A visão geral deste capítulo ainda será escrita.

## Seções

| Seção | Tópico |
|---|---|
| [8.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/01-regressao-linear-simples.html) | Regressão Linear Simples |
| [8.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/02-avaliando-o-ajuste.html) | Avaliando o Ajuste: R² e Erro |
| [8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-regressao-multipla.html) | Regressão Múltipla |
| [8.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-preditores-qualitativos.html) | Preditores Qualitativos |
| [8.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-interacao-e-termos-nao-lineares.html) | Interação e Termos Não Lineares |
| [8.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-outliers-alavancagem-e-colinearidade.html) | Outliers, Alavancagem e Colinearidade |
| [8.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/07-regressao-linear-contra-k-vizinhos.html) | Regressão Linear contra k-Vizinhos |

## Regressão Linear Simples

> **📌 Nota**
>
> Esta seção corresponde às seções 3.1 e 3.1.1 de James et al. (2023).

`Advertising` traz o quanto duzentos mercados investiram em três mídias de propaganda — televisão, rádio e jornal — e quantas unidades do produto cada um vendeu. A pergunta mais simples que esse dado permite fazer é também a primeira: o investimento em TV, sozinho, ajuda a prever vendas? Regressão linear simples responde ajustando uma reta a exatamente dois números por mercado, `tv` e `vendas`, deixando `radio` e `jornal` de fora por enquanto.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

plt.style.use("estilo-figuras.mplstyle")

### Uma reta para tv e vendas

In [ ]:
propaganda = pd.read_csv("dados/Advertising.csv")
propaganda.shape, propaganda.columns.tolist()

Duzentos mercados, quatro colunas — `tv`, `radio`, `jornal` e `vendas`, a primeira em milhares de dólares, a última em milhares de unidades. Regressão linear simples escreve a relação entre as duas que interessam aqui como

$$
\text{vendas} \approx \beta_0 + \beta_1 \cdot \text{tv}.
$$

$\beta_0$ e $\beta_1$ são duas constantes, desconhecidas até se estimar: $\beta_0$ é o **intercepto** — o valor esperado de vendas quando o investimento em TV é zero —, e $\beta_1$ é a **inclinação** — o quanto vendas muda, em média, para cada unidade a mais de tv. O símbolo "≈" marca que a relação é uma aproximação: nada garante que dois mercados com o mesmo investimento em TV vendam exatamente o mesmo, e o quanto cada um foge da reta é o que o resto desta seção mede.

### O resíduo, e a soma que ele eleva ao quadrado

Uma vez que se tem estimativas $\hat\beta_0$ e $\hat\beta_1$, a reta prevê

$$
\hat y_i = \hat\beta_0 + \hat\beta_1 x_i,
$$

e o **resíduo** daquele mercado é a distância entre o que ele de fato vendeu e o que a reta previu para o mesmo investimento em TV:

$$
e_i = y_i - \hat y_i.
$$

A **soma dos quadrados dos resíduos** (RSS) soma esse erro, ao quadrado, sobre os duzentos mercados:

$$
\text{RSS} = e_1^2 + e_2^2 + \cdots + e_n^2 = \sum_{i=1}^{n} \left(y_i - \hat\beta_0 - \hat\beta_1 x_i\right)^2.
$$

Elevar ao quadrado, em vez de somar o valor absoluto de cada resíduo, pune um erro grande desproporcionalmente mais do que vários erros pequenos, e deixa RSS como uma soma de parábolas em $\beta_0$ e $\beta_1$ — uma superfície com um único fundo, que se acha por fórmula fechada em vez de busca. Ajustar a reta é escolher, entre todos os pares $(\beta_0, \beta_1)$ possíveis, o único que minimiza essa soma: é a esse critério que se dá o nome de **mínimos quadrados**.

> **🔷 Conceito**
>
> O **resíduo** $e_i = y_i - \hat y_i$ mede a distância entre um valor observado e o previsto pela reta. A **soma dos quadrados dos resíduos** (RSS) soma $e_i^2$ sobre todas as observações. **Mínimos quadrados** é o critério que escolhe $\hat\beta_0$ e $\hat\beta_1$ minimizando RSS — nenhum outro par de coeficientes produz uma reta com RSS menor.

### A conta à mão: duas médias bastam

Minimizar RSS por cálculo — derivando em relação a $\beta_0$ e a $\beta_1$ e igualando as duas derivadas a zero — leva a uma fórmula fechada que depende só das médias de `tv` e `vendas`, e dos desvios de cada ponto em relação a elas:

$$
\hat\beta_1 = \frac{\displaystyle\sum_{i=1}^{n} (x_i - \bar x)(y_i - \bar y)}{\displaystyle\sum_{i=1}^{n} (x_i - \bar x)^2}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x.
$$

Não precisa de nenhuma biblioteca de otimização — dá para calcular direto com `numpy`:

In [ ]:
tv = propaganda["tv"]
vendas = propaganda["vendas"]

tv_media = tv.mean()
vendas_media = vendas.mean()
beta1_mao = ((tv - tv_media) * (vendas - vendas_media)).sum() / ((tv - tv_media) ** 2).sum()
beta0_mao = vendas_media - beta1_mao * tv_media

round(tv_media, 2), round(vendas_media, 2), round(beta1_mao, 4), round(beta0_mao, 4), round(beta1_mao * 1000, 1)

O investimento médio em TV é 147,04 (mil dólares); a venda média, 14,02 (mil unidades). A partir só dessas duas médias e dos desvios em relação a elas, $\hat\beta_1$ sai 0,0475 e $\hat\beta_0$, 7,0326. Como `tv` e `vendas` vêm as duas em milhares, $\hat\beta_1 \times 1.000$ traduz a inclinação para a escala do dinheiro gasto: 47,5 — cada mil dólares a mais investidos em TV está associado, em média, a 47,5 unidades a mais vendidas.

### A mesma conta, pronta: `LinearRegression`

O `scikit-learn` resolve a mesma minimização sem passar pelas médias explicitamente. `LinearRegression().fit(X, y)` recebe o preditor e a resposta e devolve um objeto já ajustado, com a inclinação em `.coef_` e o intercepto em `.intercept_` — os dois como array e escalar do `numpy`, por isso o `float(...)` ao redor de cada um daqui em diante. `X` entra como o `DataFrame` que já veio do `pandas`, mesmo sendo de uma coluna só, sem nenhum `.to_numpy()`: o estimador aceita e devolve, em `.feature_names_in_`, o nome de coluna que recebeu.

In [ ]:
X = propaganda[["tv"]]
y = propaganda["vendas"]

modelo = LinearRegression().fit(X, y)
beta1_sklearn = float(modelo.coef_[0])
beta0_sklearn = float(modelo.intercept_)

(
    modelo.feature_names_in_,
    (round(beta0_mao, 4), round(beta1_mao, 4)),
    (round(beta0_sklearn, 4), round(beta1_sklearn, 4)),
    bool(np.allclose([beta0_mao, beta1_mao], [beta0_sklearn, beta1_sklearn])),
)

`feature_names_in_` guarda só `tv`, o único nome que o `DataFrame` de uma coluna carregava. Os dois pares de coeficiente — o calculado à mão e o que saiu do `.fit()` — são (7,0326; 0,0475) nos dois casos, e `np.allclose` confirma: `True`. É a mesma fórmula fechada por trás das duas contas; a segunda só evita escrever as médias à mão.

### A reta, e o resíduo que ela deixa

In [ ]:
# Figura: Duzentos mercados: vendas contra o investimento em TV, com a reta de mínimos quadrados por cima. Cada segmento liga um mercado observado à previsão da reta para o mesmo investimento — o resíduo daquele mercado, o e_i que RSS eleva ao quadrado.
yhat = modelo.predict(X)
grade_tv = np.linspace(tv.min(), tv.max(), 200)
reta_grade = modelo.predict(pd.DataFrame({"tv": grade_tv}))

fig, ax = plt.subplots()
ax.vlines(tv, np.minimum(vendas, yhat), np.maximum(vendas, yhat), color="C1", linewidth=1)
ax.plot(grade_tv, reta_grade, color="C0", linewidth=2, label="reta ajustada")
ax.scatter(tv, vendas, color="C2", s=18, zorder=3, label="observado")
ax.set_xlabel("tv (milhares de dólares)")
ax.set_ylabel("vendas (milhares de unidades)")
ax.legend()
plt.tight_layout()
plt.show()

A reta captura a tendência — vender mais conforme se investe mais em TV —, mas nenhum mercado senta exatamente sobre ela: sempre sobra um segmento. Somar o quadrado dos duzentos segmentos desta figura dá exatamente o RSS que a reta minimiza:

In [ ]:
rss_min = float(((vendas - yhat) ** 2).sum())
round(rss_min, 2)

2.102,53 — nenhuma outra reta, entre todos os pares $(\beta_0, \beta_1)$ possíveis, soma menos que isso.

### Um vale com um fundo só

RSS, como soma de quadrados de uma função linear de $\beta_0$ e $\beta_1$, é uma superfície convexa nesses dois parâmetros: um paraboloide elíptico, sem platôs nem mínimos locais além de um único ponto. Variando $\beta_0$ e $\beta_1$ numa grade ao redor de $(\hat\beta_0, \hat\beta_1)$ e calculando RSS em cada combinação, essa forma aparece em curvas de nível — cada uma liga os pares que produzem o mesmo RSS.

In [ ]:
# Figura: Curvas de nível de RSS sobre (β0, β1), na regressão de vendas sobre tv em Advertising. O ponto marcado é (β̂0, β̂1); cada curva liga pares com o mesmo RSS, e as seis se fecham ao redor desse único ponto — não há outro vale na janela.
grade_beta0 = np.linspace(3.5, 10.5, 300)
grade_beta1 = np.linspace(0.02, 0.075, 300)
malha_beta0, malha_beta1 = np.meshgrid(grade_beta0, grade_beta1)

tv_np = tv.to_numpy()
vendas_np = vendas.to_numpy()
residuo_grade = vendas_np - malha_beta0[:, :, None] - malha_beta1[:, :, None] * tv_np
rss_grade = (residuo_grade ** 2).sum(axis=2)

niveis_rss = np.array([2150.0, 2200.0, 2300.0, 2400.0, 2500.0, 2600.0])

fig, ax = plt.subplots()
contornos = ax.contour(malha_beta0, malha_beta1, rss_grade, levels=niveis_rss, cmap="Blues")
ax.scatter([beta0_sklearn], [beta1_sklearn], color="C1", s=40, zorder=3)
ax.annotate(
    r"$(\hat\beta_0,\ \hat\beta_1)$",
    xy=(beta0_sklearn, beta1_sklearn),
    xytext=(10, -14),
    textcoords="offset points",
    fontsize=9,
)
ax.set_xlabel(r"$\beta_0$")
ax.set_ylabel(r"$\beta_1$")
barra = fig.colorbar(contornos, ax=ax, shrink=0.9, pad=0.02)
barra.set_label("RSS")
plt.tight_layout()
plt.show()

A legenda promete curvas fechadas, então a afirmação se confere contando, não olhando: o próprio objeto que o `contour` devolve guarda, para cada nível, os segmentos de linha que desenhou, e um segmento fecha quando termina no mesmo ponto em que começou.

In [ ]:
segmentos_totais = 0
segmentos_fechados = 0
for segmentos_do_nivel in contornos.allsegs:
    for segmento in segmentos_do_nivel:
        if len(segmento) == 0:
            continue
        segmentos_totais += 1
        if np.allclose(segmento[0], segmento[-1]):
            segmentos_fechados += 1

segmentos_totais, segmentos_fechados, segmentos_fechados == segmentos_totais

Seis níveis, seis segmentos desenhados, e os seis fecham: `segmentos_totais` e `segmentos_fechados` saem iguais, 6 e 6. Nenhuma curva sai cortada pela borda da janela — o que confirma, em vez de supor, que RSS tem um único vale nesta vizinhança de $(\hat\beta_0, \hat\beta_1)$.

In [ ]:
minimo_real_menor_que_grade = bool(rss_min < rss_grade.min())
round(rss_min, 2), round(float(rss_grade.min()), 2), minimo_real_menor_que_grade

O mínimo verdadeiro, 2.102,53, fica abaixo até do menor valor que a própria grade alcança, 2.102,56 — `minimo_real_menor_que_grade` é `True`. Nenhum dos 300×300 pontos testados coincide exatamente com $(\hat\beta_0, \hat\beta_1)$, só passa perto: é a fórmula fechada, não a grade, que encontra o fundo do vale de verdade. RSS diz qual par de coeficientes é o melhor entre os que este dado observou; não diz se essa reta presta para prever vendas em geral, nem quanto de vendas ela de fato explica.

## Avaliando o Ajuste: R² e Erro

> **📌 Nota**
>
> Esta seção corresponde à seção 3.1.3 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Regressão Múltipla

> **📌 Nota**
>
> Esta seção corresponde à seção 3.2 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Preditores Qualitativos

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.1 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Interação e Termos Não Lineares

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.2 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Outliers, Alavancagem e Colinearidade

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.3 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Regressão Linear contra k-Vizinhos

> **📌 Nota**
>
> Esta seção corresponde à seção 3.5 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Leituras adicionais

*A escrever.*

## Referências

- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.